# Chapter 7 companion: data reconstruction and data holds.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Reproduces:
  * the zero-order, first-order and fractional-order holds as transfer functions
      G_ho(s) = (1 - e^{-Ts})/s
      G_h1(s) = (1 + Ts)/T [(1 - e^{-Ts})/s]^2
      G_hb(s) = (1 - b e^{-Ts})(1 - e^{-Ts})/s + b/(T s^2) (1 - e^{-Ts})^2
    and their gain and phase at wT = 0.5 and at ws/2 (the comparison table);
  * where the ZOH and FOH lags are equal (wT = 2.33, 0.37 ws) and the peak gain
    of the FOH (about 1.59 T near 0.34 ws);
  * the images of a held sinusoid at ws/10: amplitudes 0.984, 0.109, 0.089;
  * the three holds reconstructing one sampled signal (figure).

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import brentq

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)
T = 1.0                         # results are quoted in units of T
ws = 2 * np.pi / T

In [ ]:
def G_zoh(w):
    s = 1j * w
    return (1 - np.exp(-T * s)) / s

In [ ]:
def G_frac(w, beta):
    s = 1j * w
    x = np.exp(-T * s)
    return (1 - beta * x) * (1 - x) / s + beta / (T * s**2) * (1 - x) ** 2

In [ ]:
def G_foh(w):
    s = 1j * w
    return (1 + T * s) / T * ((1 - np.exp(-T * s)) / s) ** 2

In [ ]:
def gain_phase(G):
    return abs(G) / T, np.degrees(np.angle(G))

In [ ]:
print("Gain (in units of T) and phase (deg)")
print("                     wT = 0.5              w = ws/2")

In [ ]:
for name, f in [("ZOH          ", G_zoh), ("beta = 0.5   ", lambda w: G_frac(w, 0.5)),
                ("FOH (beta=1) ", G_foh)]:
    g1, p1 = gain_phase(f(0.5 / T))
    g2, p2 = gain_phase(f(np.pi / T))
    p2 = p2 - 360 if p2 > 0 else p2           # report the lag as a negative angle
    print(f"  {name}   {g1:.3f} T, {p1:6.1f}       {g2:.3f} T, {p2:7.1f}")

In [ ]:
print("  the fractional hold with beta = 1 equals the FOH:",
      np.allclose(G_frac(np.array([0.3, 1.7]), 1.0), G_foh(np.array([0.3, 1.7]))))

In [ ]:
# equal lags: the ZOH lags by wT/2, the FOH by wT - atan(wT)
x = brentq(lambda v: v / 2 - (v - np.arctan(v)), 0.5, 3.0)

In [ ]:
print(f"\nEqual phase lag where wT/2 = wT - atan(wT): wT = {x:.2f}, w = {x / (2 * np.pi):.2f} ws")
w = np.linspace(0.01, ws, 200001)
g = np.abs(G_foh(w)) / T
print(f"Peak gain of the FOH: {g.max():.2f} T at w = {w[g.argmax()] / ws:.2f} ws")

In [ ]:
# a held sinusoid at w0 = ws/10: components at |w0 + n ws| with amplitude |sinc(pi (w0 + n ws)/ws)|
w0 = ws / 10

In [ ]:
print("\nZOH on sin(w0 t), w0 = ws/10: amplitude of the component at |w0 + n ws|")

In [ ]:
for n in (0, -1, 1):
    v = np.pi * (w0 + n * ws) / ws
    print(f"  n = {n:2d}, |w0 + n ws| = {abs(w0 + n * ws) / ws:.1f} ws: {abs(np.sin(v) / v):.3f}")

In [ ]:
# low-frequency delay of the fractional hold: (1 - beta) T/2
for beta in (0.0, 0.5, 1.0):
    wl = 1e-3
    print(f"  beta = {beta}: low-frequency delay = {-np.angle(G_frac(wl, beta)) / wl:.4f} T"
          f" (formula (1 - beta)/2 = {(1 - beta) / 2:.4f})")

In [ ]:
# ---------------------------------------------------------------- reconstruction figure
Ts = 0.5

In [ ]:
f = lambda t: 1.1 + 0.8 * np.sin(0.9 * t) + 0.12 * t
tk = Ts * np.arange(0, 21)
ek = f(tk)
t = np.linspace(0, 10, 2001)
k = np.minimum((t / Ts).astype(int), len(tk) - 1)
zoh = ek[k]
prev = ek[np.maximum(k - 1, 0)]
slope = (ek[k] - prev) / Ts
foh = ek[k] + slope * (t - tk[k])
half = ek[k] + 0.5 * slope * (t - tk[k])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(t, f(t), "k", lw=1, label="e(t)")
axes[0].plot(t, zoh, label="ZOH")
axes[0].plot(t, half, label="beta = 0.5")
axes[0].plot(t, foh, label="FOH")
axes[0].plot(tk, ek, "ko", ms=3)
axes[0].set(xlabel="t (s)", title="Reconstruction from samples, T = 0.5 s")
axes[0].legend(fontsize=8)
wn = np.linspace(0.001, 2 * ws, 2000)
axes[1].plot(wn / ws, np.abs(G_zoh(wn)) / T, label="ZOH")
axes[1].plot(wn / ws, np.abs(G_foh(wn)) / T, label="FOH")
axes[1].set(xlabel="omega / omega_s", ylabel="|G_h| / T", title="Hold magnitudes")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch07_holds.png", dpi=150)
print("\nSaved figures/ch07_holds.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Holds on a sine.** The samples of $e(t) = \sin t$ (so $\omega = 1$ rad/s and $\omega T = T$)
are rebuilt by the zero-order hold and by the fractional-order hold
$\bar e(t) = e(kT) + \beta\,[e(kT) - e((k-1)T)](t - kT)/T$: $\beta = 0$ is the ZOH,
$\beta = 1$ the FOH. At $T = 0.5$ s the ZOH lags by $14.3^\circ$ and the FOH by only $2.1^\circ$.
Grow $T$ and watch the staircase fall behind, while the FOH overshoots where the sine bends;
move $\beta$ to trade lag against overshoot.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
W0 = 1.0          # rad/s
T_END = 12.0      # seconds shown


def _G_hold(w, T, beta):
    """G_hb(jw) = (1 - b e^{-jwT})(1 - e^{-jwT})/(jw) + b/(T (jw)^2) (1 - e^{-jwT})^2."""
    s = 1j * w
    x = np.exp(-T * s)
    return (1 - beta * x) * (1 - x) / s + beta / (T * s ** 2) * (1 - x) ** 2


def _gain_phase(T, beta):
    """Gain over T and phase in degrees at w = W0, phase unwrapped from w = 0."""
    w = np.linspace(1e-4, W0, 400)
    G = _G_hold(w, T, beta)
    return abs(G[-1]) / T, np.degrees(np.unwrap(np.angle(G))[-1]) + 0.0


def _hold(t, T, beta):
    k = np.floor(t / T + 1e-12)
    ek = np.sin(W0 * k * T)
    ekm1 = np.where(k >= 1, np.sin(W0 * (k - 1) * T), 0.0)   # e = 0 before t = 0
    return ek + beta * (ek - ekm1) * (t - k * T) / T


def explore(T=0.5, beta=1.0):
    T, beta = round(T, 10), round(beta, 10)
    t = np.linspace(0, T_END, 3001)
    e = np.sin(W0 * t)
    zoh = _hold(t, T, 0.0)
    frac = _hold(t, T, beta)
    tk = T * np.arange(int(T_END / T) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.0),
                                   gridspec_kw={"width_ratios": [1.8, 1]})
    ax1.plot(t, e, color=GRAY, lw=1.4, label=r"$e(t) = \sin t$")
    ax1.plot(t, zoh, color=BLUE, lw=1.4, label="ZOH")
    if beta > 0:
        ax1.plot(t, frac, color=ORANGE, lw=1.8,
                 label="FOH" if beta == 1 else rf"fractional, $\beta$ = {beta:g}")
    ax1.plot(tk, np.sin(W0 * tk), "o", color="k", ms=3.5, label="samples")
    ax1.set_xlim(0, T_END)
    lo = min(-1.3, frac.min() - 0.1)
    hi = max(1.3, frac.max())
    ax1.set_ylim(lo, hi + 0.25 * (hi - lo) + 0.2)          # room for the legend
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel(r"$\bar e(t)$")
    ax1.set_title(rf"T = {T:g} s ($\omega T$ = {W0 * T:g}), $\beta$ = {beta:g}", fontsize=11)
    ax1.legend(fontsize=10, loc="upper right", ncol=4)

    ws = 2 * np.pi / T
    w = np.linspace(1e-4, 2 * ws, 1500)
    ax2.plot(w / ws, np.abs(_G_hold(w, T, 0.0)) / T, color=BLUE, lw=1.6, label="ZOH")
    if beta > 0:
        ax2.plot(w / ws, np.abs(_G_hold(w, T, beta)) / T, color=ORANGE, lw=1.8,
                 label=rf"$\beta$ = {beta:g}")
    ax2.axvline(W0 / ws, color=TEAL, ls="--", lw=1.4, label=r"$\omega = 1$ rad/s")
    ax2.set_xlim(0, 2)
    ax2.set_ylim(0, 1.75)
    ax2.set_xlabel(r"$\omega / \omega_s$")
    ax2.set_ylabel(r"$|G_h(j\omega)| / T$")
    ax2.set_title("hold gain", fontsize=11)
    ax2.legend(fontsize=10, loc="upper right")
    fig.tight_layout()

    g0, ph0 = _gain_phase(T, 0.0)
    gb, phb = _gain_phase(T, beta)
    print(f"omega T = {W0 * T:g}  (omega = {W0 * T / (2 * np.pi):.3f} omega_s)")
    print(f"ZOH:          gain {g0:.3f} T, phase {round(ph0, 1) + 0.0:6.1f} deg")
    print(f"beta = {beta:<4g}   gain {gb:.3f} T, phase {round(phb, 1) + 0.0:6.1f} deg;  "
          f"low-frequency delay (1 - beta) T/2 = {(1 - beta) * T / 2:.3f} s")
    after = t >= 2 * T
    print(f"largest |e(t) - hold output| after t = 2T: ZOH {np.max(np.abs(e - zoh)[after]):.3f}, "
          f"beta = {beta:g}: {np.max(np.abs(e - frac)[after]):.3f}")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             T=FloatSlider(value=0.5, min=0.1, max=2.5, step=0.05, description="T (s)",
                           continuous_update=False),
             beta=FloatSlider(value=1.0, min=0.0, max=1.0, step=0.1, description="beta",
                              continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 7.** These cells go with the Hands-on exercises at the end of
Chapter 7. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def staircase(fs, f, t, phase=0.0):
    """Zero-order hold of the samples of sin(2 pi f t + phase) taken at fs."""
    return np.sin(2 * np.pi * f * np.floor(t * fs) / fs + phase)

In [ ]:
def line_amplitudes(x, dt, freqs):
    """Amplitude of the sinusoidal component of x at each frequency (whole periods assumed)."""
    t = np.arange(len(x)) * dt
    return np.array([2 * abs(np.mean(x * np.exp(-2j * np.pi * fr * t))) for fr in freqs])

In [ ]:
# ---------------------------------------------------------------- H1
def h1_staircase_spectrum(f=4.0, fs=100.0):
    """H1: a held sine; its fundamental and the images at n fs +- f."""
    dt = 1 / (fs * 200)
    t = np.arange(0, 1.0, dt)                     # 1 s: whole periods of f and of fs
    ub = staircase(fs, f, t)
    lines = [f] + [n * fs + s * f for n in (1, 2) for s in (-1, 1)]
    amps = line_amplitudes(ub, dt, lines)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [3, 2]})
    show = t < 2 / f
    ax1.plot(t[show], np.sin(2 * np.pi * f * t[show]), color=GRAY, lw=1.2, label="sine")
    ax1.plot(t[show], ub[show], color=ORANGE, lw=1.4, label="staircase")
    ax1.set_xlabel("t (s)")
    ax1.legend(loc="upper right", fontsize=9)
    ax1.set_title(f"f = {f:g} Hz, fs = {fs:g} Hz", fontsize=11)
    ax2.stem(lines, amps, linefmt=BLUE, markerfmt="o", basefmt=" ")
    ax2.set_yscale("log")
    ax2.set_ylim(1e-3, 2)
    ax2.set_xlabel("frequency (Hz)")
    ax2.set_title(f"image/fundamental {amps[1] / amps[0]:.4f}; f/(fs-f) = {f / (fs - f):.4f}",
                  fontsize=10)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_pitch_lag(fs=20.0, wc=3.0, pm=50.0):
    """H2: sampler + ZOH seen as a gain and a phase at the crossover frequency."""
    T = 1 / fs
    w = np.linspace(0.01, 2 * np.pi * fs, 2000)
    x = w * T / 2
    gain = np.abs(np.sin(x) / x)                  # sampler (1/T) times hold (T sinc)
    lag = np.degrees(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.6))
    ax1.plot(w, gain, color=BLUE, lw=1.6)
    ax1.axvline(wc, color=ORANGE, ls="--", lw=1)
    ax1.set_xlabel("omega (rad/s)")
    ax1.set_title("gain of sampler + ZOH", fontsize=11)
    ax2.plot(w, lag, color=TEAL, lw=1.6)
    ax2.axvline(wc, color=ORANGE, ls="--", lw=1)
    ax2.set_xlabel("omega (rad/s)")
    ax2.set_title(f"lag at wc: {np.degrees(wc * T / 2):.2f} deg, PM about "
                  f"{pm - np.degrees(wc * T / 2):.1f} deg", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_flexible_arm(f=8.0, fs=50.0):
    """H3: held sine into a lightly damped 45 Hz mode; largest fast component of the tip motion."""
    wr, z = 2 * np.pi * 45, 0.01
    arm = signal.lti([wr ** 2], [1, 2 * z * wr, wr ** 2])
    dt = 1 / (fs * 40)
    t = np.arange(0, 8.0, dt)
    ub = staircase(fs, f, t)
    _, y, _ = signal.lsim(arm, ub, t, interp=False)
    keep = t >= 6.0                              # last 2 s, after the mode has settled
    tk, yk = t[keep], y[keep]
    spec = 2 * np.abs(np.fft.rfft(yk)) / len(yk)
    freqs = np.fft.rfftfreq(len(yk), dt)
    fast = freqs > 20
    i = np.argmax(spec * fast)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    show = tk < 6.4
    ax.plot(tk[show], ub[keep][show], color=GRAY, lw=1, label="held command")
    ax.plot(tk[show], yk[show], color=ORANGE, lw=1.2, label="tip position")
    ax.set_xlabel("t (s)")
    ax.set_ylim(-8, 8)
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"f = {f:g} Hz, fs = {fs:g} Hz: largest fast component {spec[i]:.2f} A "
                 f"at {freqs[i]:.1f} Hz", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_fractional_hold(beta=0.45, sigma=1.0):
    """H4: fractional-order hold; low-frequency delay, rms of held noise, step overshoot."""
    rng = np.random.default_rng(1)
    n, m = 4000, 50                               # samples, points per period
    eps = sigma * rng.standard_normal(n)
    x = np.arange(m) / m
    held = [(1 + beta * x) * eps[k] - beta * x * eps[k - 1] for k in range(1, n)]
    rms = np.sqrt(np.mean(np.concatenate(held) ** 2))
    u = np.array([0.0, 0, 1, 1, 1, 1])
    xe = np.linspace(0, 1, m)                     # up to the instant just before the next sample
    step = np.concatenate([u[k] + beta * (u[k] - u[k - 1]) * xe for k in range(1, len(u))])
    ts = np.arange(len(step)) / m
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(ts, step, color=ORANGE, lw=1.6, label="held step")
    ax.axhline(1.5, color=TEAL, ls="--", lw=1, label="50% overshoot limit")
    ax.set_xlabel("t / T")
    ax.set_ylim(-0.1, 2.2)
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"beta = {beta:.2f}: delay {(1 - beta) / 2:.3f} T, rms {rms / sigma:.3f} sigma "
                 f"(formula {np.sqrt(1 + beta + 2 * beta ** 2 / 3):.3f}), overshoot "
                 f"{100 * (step.max() - 1):.0f}%", fontsize=10)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(h1_staircase_spectrum,
             f=FloatSlider(value=4, min=1, max=25, step=1, description="f (Hz)",
                           continuous_update=False),
             fs=FloatSlider(value=100, min=50, max=400, step=50, description="fs (Hz)",
                            continuous_update=False))
    interact(h2_pitch_lag, fs=FloatSlider(value=20, min=2, max=50, step=0.5, description="fs (Hz)",
                                          continuous_update=False),
             wc=FloatSlider(value=3, min=0.5, max=10, step=0.5, description="wc (rad/s)",
                            continuous_update=False),
             pm=FloatSlider(value=50, min=20, max=80, step=5, description="PM (deg)",
                            continuous_update=False))
    interact(h3_flexible_arm, f=FloatSlider(value=8, min=1, max=20, step=0.5, description="f (Hz)",
                                            continuous_update=False),
             fs=FloatSlider(value=50, min=30, max=80, step=5, description="fs (Hz)",
                            continuous_update=False))
    interact(h4_fractional_hold, beta=FloatSlider(value=0.45, min=0, max=1, step=0.05,
                                                  description="beta", continuous_update=False),
             sigma=FloatSlider(value=1, min=0.5, max=2, step=0.5, description="sigma",
                               continuous_update=False))
except ImportError:
    h1_staircase_spectrum()
    h2_pitch_lag()
    h3_flexible_arm()
    h4_fractional_hold()